In [1]:
import os 
from dotenv import load_dotenv

#langchain 

from langchain_community.document_loaders import TextLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.embeddings import JinaEmbeddings


from langchain_community.vectorstores import FAISS

C:\Users\ayush\AppData\Local\Temp\ipykernel_16740\3769572910.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
load_dotenv()

True

In [3]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

Loading Data

In [4]:
DATA_FILE_PATH = os.path.join("data",'hr_policy.txt')

Data ingestion

In [5]:
loader = TextLoader(DATA_FILE_PATH, encoding = 'utf-8') 

documents = loader.load() 



Langchain document


In [21]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

Splitting our data

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
) 

chunks = text_splitter.split_documents(documents)

Embedding our data


In [23]:
embeddings_model = JinaEmbeddings(model_name='jina-embeddings-v2-base-en')

Store data in vector DB


In [29]:
vector_store = FAISS.from_documents(chunks, embeddings_model)

In [35]:
vector_store.index.ntotal

9

In [36]:
test_query = "How many sick leaves employees get" 

top_matches = vector_store.similarity_search(test_query,k=2)

In [49]:
print(f"Query: {test_query}") 

for i,match in enumerate(top_matches):
    print(f"Match {i+1}: {match.page_content}")
    print()

Query: How many sick leaves employees get
Match 1: 1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

Match 2: 7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



DATA Retrieval

In [40]:
from langchain_groq import ChatGroq



llm = ChatGroq(model="openai/gpt-oss-120b", temperature =0) 



Tool

In [41]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [45]:
print(search_hr_policy("How many leaves?"))

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.

4. NOTICE PERIOD
Employees who wish to resign must serve a notice period of 30 days.
During the probation period, the notice period is reduced to 15 days.
The company may waive the notice period at its discretion, with full and final settlement
processed within 45 days of the last working day.


AI AGENT

In [42]:
from langchain.agents import create_agent
tools = [search_hr_policy]
hr_assistant = create_agent(
    model=llm,
    tools = tools,
    system_prompt="""

    You are a friendly HR assistant.
    Always use the search_hr_policy tool to look up
    facts before answering.
    If the answer isn't in the search results, say you don't know
    "instead of guessing".

"""
)

In [43]:
def ask_hr_assistant(question:str) -> str:
    print("*"*60)
    print("Question: ",question) 
    print("*"*60)

    response = hr_assistant.invoke({"messages":[{"role":"user","content":question}]})
    answer = response['messages'][-1].content

    print("Answer:",answer)
    print("="*60)

In [104]:
ask_hr_assistant("tell me about leave policy")

************************************************************
Question:  tell me about leave policy
************************************************************
Answer: Here’s a quick rundown of our **Leave Policy**:

| Type of Leave | Entitlement | Key Details |
|---------------|------------|-------------|
| **Annual (Paid) Leave** | **20 days per calendar year** | • Submit your request through the HR portal **at least 5 working days** before the intended start date.<br>• Unused days can be carried forward **up to 5 days** into the next year. |
| **Sick Leave** | **10 paid days per year** | • Separate from annual leave.<br>• If you’re off for **more than 2 consecutive days**, you’ll need to provide a medical certificate. |
| **Carry‑Forward Limit** | Up to **5 days** of annual leave | Any excess beyond the 5‑day limit lapses at year‑end. |

**How to Apply**  
1. Log into the HR portal.  
2. Choose the leave type (Annual or Sick).  
3. Enter the dates and any supporting notes (e.g., med